# 3. Convex sets  [CO 2–5]

TOML-MIRI — Maximum Likelihood Estimation and Convex Optimization, exploration exercises.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, fixed

np.set_printoptions(precision=4, suppress=True)
import plotly.graph_objects as go
from scipy.spatial import ConvexHull

def check(label, ok):
    print(f"{'✓' if ok else '✗'}  {label}")

---
## 3.1 Convexity test with segments

**Task.** Write a function that samples pairs of points $x, y$ of a 2-D set $C$ (given as a membership test) and checks whether $\theta x + (1-\theta)y \in C$ for $\theta\in[0,1]$. Plot the set and, in red, the segments that leave it. Test: a disc, an annulus, a crescent, the union of two discs, and the intersection of two discs.

### Prediction

- Disc and intersection of two discs: no red segments. Annulus: segments crossing the hole. Crescent: segments across the "bite". Union of two separated discs: segments across the gap between the discs.

In [ ]:
def sample_in(member, n, box=(-2.5, 2.5), rng=None):
    rng = rng or np.random.default_rng(0)
    pts = []
    while len(pts) < n:
        P = rng.uniform(*box, size=(4 * n, 2))
        pts.extend(P[member(P)])
    return np.array(pts[:n])

def convexity_test(member, n_pairs=2000, n_theta=60, box=(-2.5, 2.5), seed=0):
    # returns the sampled pairs and a boolean 'leaves C' for each pair
    rng = np.random.default_rng(seed)
    X = sample_in(member, n_pairs, box, rng); Yp = sample_in(member, n_pairs, box, rng)
    th = np.linspace(0, 1, n_theta)[:, None, None]
    segs = th * X[None] + (1 - th) * Yp[None]                # (n_theta, n_pairs, 2)
    inside = member(segs.reshape(-1, 2)).reshape(n_theta, n_pairs)
    return X, Yp, ~inside.all(axis=0)

def plot_test(ax, member, title, box=(-2.5, 2.5), **kw):
    X, Yp, bad = convexity_test(member, box=box, **kw)
    g = np.linspace(*box, 400); GX, GY = np.meshgrid(g, g)
    ax.contourf(GX, GY, member(np.column_stack([GX.ravel(), GY.ravel()])).reshape(GX.shape),
                levels=[0.5, 1.5], colors=["#cfe0f3"])
    for a, b in zip(X[bad][:40], Yp[bad][:40]):
        ax.plot(*np.column_stack([a, b]), "r-", lw=0.8)
    ax.set_aspect("equal"); ax.set_xlim(*box); ax.set_ylim(*box)
    ax.set_title(f"{title}\n{bad.mean():.1%} of {bad.size} segments leave the set")
    return bad.mean()

def disc(c, r):
    c = np.asarray(c)
    return lambda P: np.sum((P - c) ** 2, axis=1) <= r**2

sets = {
    "disc": disc((0, 0), 2),
    "annulus": lambda P: (np.sum(P**2, axis=1) <= 4) & (np.sum(P**2, axis=1) >= 1),
    "crescent": lambda P: disc((0, 0), 2)(P) & ~disc((0.9, 0), 1.6)(P),
    "union of two discs": lambda P: disc((-1.2, 0), 1)(P) | disc((1.2, 0), 1)(P),
    "intersection of two discs": lambda P: disc((-0.7, 0), 1.6)(P) & disc((0.7, 0), 1.6)(P),
}
fig, axes = plt.subplots(1, 5, figsize=(24, 5))
fail = {name: plot_test(ax, mem, name) for ax, (name, mem) in zip(axes, sets.items())}
plt.show()

### Verify numerically

In [ ]:
check("disc: no segment leaves the set", fail["disc"] == 0)
check("intersection of two discs: no segment leaves the set", fail["intersection of two discs"] == 0)
check("annulus, crescent and union: some segments leave the set",
      all(fail[k] > 0 for k in ["annulus", "crescent", "union of two discs"]))
# [CO 3.1] with random convex sets: intersections of random discs/halfspaces are always convex, unions usually not
rng = np.random.default_rng(5)
inter_ok, union_ok = [], []
for _ in range(20):
    a, b = disc(rng.uniform(-1, 1, 2), rng.uniform(0.8, 1.6)), disc(rng.uniform(-1, 1, 2), rng.uniform(0.8, 1.6))
    if sample_in(lambda P: a(P) & b(P), 1, rng=rng).size == 0:
        continue
    inter_ok.append(convexity_test(lambda P: a(P) & b(P), n_pairs=400, seed=1)[2].mean() == 0)
    union_ok.append(convexity_test(lambda P: a(P) | b(P), n_pairs=400, seed=1)[2].mean() == 0)
check(f"intersection of two random discs convex in {sum(inter_ok)}/{len(inter_ok)} trials", all(inter_ok))
print(f"union of two random discs convex in {sum(union_ok)}/{len(union_ok)} trials (only when one disc contains the other)")

---
### Explain

**1. Which sets fail and why?**

The **annulus**, the **crescent** and the **union of two discs** fail; the disc and the intersection of two discs pass (0% of segments leave). The annulus fails because it has a hole: a segment between two points on opposite sides of the inner circle passes through the hole. The crescent is a disc with a disc-shaped bite removed; its boundary is concave on the inner side, so chords between the two horns cross the bite. The union fails whenever the two discs are not nested: a segment from one disc to the other crosses the gap (or the "waist") between them. Note that a sampling test can only *refute* convexity (one red segment is a proof of non-convexity); passing it is evidence, not a proof.

**2. Use your tool to illustrate [CO 3.1]: why is an intersection of convex sets convex, but not a union?**

If $x, y \in L\cap M$, then $x, y\in L$ and $L$ is convex, so the whole segment is in $L$; by the same argument it is in $M$; hence it is in $L\cap M$. The argument needs *both* endpoints in *each* set, which is exactly what membership in the intersection gives. For a union, $x$ may be only in $L$ and $y$ only in $M$, and then nothing forces the segment to stay in either. The random trials confirm it: the intersection of two random discs passed every time, while the union passed only in the cases where one disc happened to contain the other (then the union is just the larger disc).

---
## 3.2 Examples of convex sets

**Task.** Plot in 2-D (and, where easy, in 3-D): a halfspace $\{a^\top x \ge b\}$, a polyhedron $\{Ax \ge b\}$, the simplex $\{x\ge0, \mathbf 1^\top x = 1\}$ in $\mathbb R^3$, the convex hull of random points (`scipy.spatial.ConvexHull`), an ellipsoid $\{\frac12x^\top Qx + p^\top x + c \le 1\}$, and the norm balls $\{\|x\|_p\le1\}$ for $p = 1, 2, \infty$.

### Prediction

- All of these are convex (halfspaces, intersections of halfspaces, hulls, sublevel sets of convex quadratics and of norms).
- The "$p = 1/2$ ball" is a star with concave sides: not convex. With $Q$ indefinite the "ellipsoid" becomes the region between the branches of a hyperbola: not convex.

In [ ]:
rng = np.random.default_rng(0)
a_h, b_h = np.array([1.0, 2.0]), 1.0
A_p = np.array([[1, 0], [0, 1], [-1, -1], [-1, 0.5]]); b_p = np.array([-1, -1, -1.5, -1.5])   # Ax ≥ b
pts = rng.normal(size=(15, 2)); hull = ConvexHull(pts)
Q_e = np.array([[2.0, 0.8], [0.8, 1.0]]); p_e = np.array([0.3, -0.2]); c_e = 0.0

def pnorm_ball(p):
    if p == np.inf:
        return lambda P: np.max(np.abs(P), axis=1) <= 1
    return lambda P: np.sum(np.abs(P) ** p, axis=1) ** (1 / p) <= 1

def in_hull(P, hull=hull):
    return np.all(P @ hull.equations[:, :2].T + hull.equations[:, 2] <= 1e-12, axis=1)

sets2d = {
    "halfspace {aᵀx ≥ b}": lambda P: P @ a_h >= b_h,
    "polyhedron {Ax ≥ b}": lambda P: np.all(P @ A_p.T >= b_p, axis=1),
    "convex hull": in_hull,
    "ellipsoid ½xᵀQx + pᵀx + c ≤ 1": lambda P: 0.5 * np.einsum("ij,jk,ik->i", P, Q_e, P) + P @ p_e + c_e <= 1,
    "‖x‖₁ ≤ 1": pnorm_ball(1), "‖x‖₂ ≤ 1": pnorm_ball(2), "‖x‖∞ ≤ 1": pnorm_ball(np.inf), "‖x‖_½ ≤ 1": pnorm_ball(0.5),
}

def show_set(ax, member, title, box=(-2.5, 2.5), extra=None):
    g = np.linspace(*box, 401); GX, GY = np.meshgrid(g, g)
    Z = member(np.column_stack([GX.ravel(), GY.ravel()])).reshape(GX.shape)
    ax.contourf(GX, GY, Z, levels=[0.5, 1.5], colors=["#cfe0f3"]); ax.contour(GX, GY, Z, levels=[0.5], colors="k", linewidths=1)
    if extra: extra(ax)
    ax.set_aspect("equal"); ax.set_title(title, fontsize=10); ax.axhline(0, c="gray", lw=0.4); ax.axvline(0, c="gray", lw=0.4)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, (name, mem) in zip(axes.ravel(), sets2d.items()):
    show_set(ax, mem, name, extra=(lambda a: a.plot(*pts.T, "k.", ms=5)) if name == "convex hull" else None)
plt.show()

# the simplex in R³
fig = go.Figure([go.Mesh3d(x=[1, 0, 0], y=[0, 1, 0], z=[0, 0, 1], opacity=0.6, color="royalblue", name="simplex"),
                 go.Scatter3d(x=[0, 1, 0, 0, 0, 0, 0], y=[0, 0, 0, 1, 0, 0, 0], z=[0, 0, 0, 0, 0, 0, 1],
                              mode="lines", line=dict(color="black"), name="axes")])
fig.update_layout(title="simplex {x ≥ 0, 1ᵀx = 1} in R³", height=450, scene=dict(aspectmode="cube"))
fig.show()

### Verify numerically with the segment test of 3.1

In [ ]:
results = {name: convexity_test(mem, n_pairs=1500)[2].mean() for name, mem in sets2d.items()}
for name, frac in results.items():
    print(f"{name:<34} fraction of segments leaving the set: {frac:.3%}")
check("all sets except the p = ½ 'ball' pass the test", all(v == 0 for k, v in results.items() if "½" not in k))
check("the p = ½ 'ball' fails", results["‖x‖_½ ≤ 1"] > 0)
check("Q of the ellipsoid is PD", np.all(np.linalg.eigvalsh(Q_e) > 0))

### Explore: an affine map $x\mapsto Ax + d$, and an indefinite $Q$

In [ ]:
A_map = np.array([[1.0, 0.8], [-0.3, 0.6]]); d_map = np.array([0.5, -0.4])
A_inv = np.linalg.inv(A_map)
def image(member):   # x' ∈ A C + d  ⇔  A⁻¹(x' − d) ∈ C  (A invertible)
    return lambda P: member((P - d_map) @ A_inv.T)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, (name, mem) in zip(axes.ravel(), sets2d.items()):
    show_set(ax, image(mem), f"A·({name}) + d", box=(-3.5, 3.5))
plt.show()
img = {name: convexity_test(image(mem), n_pairs=800, box=(-3.5, 3.5))[2].mean() for name, mem in sets2d.items()}
check("affine images of the convex sets are convex", all(v == 0 for k, v in img.items() if "½" not in k))

@interact(q12=FloatSlider(min=-3, max=3, step=0.1, value=2.0), q22=FloatSlider(min=-2, max=2, step=0.1, value=1.0))
def explore_quadric(q12, q22):
    Qx = np.array([[2.0, q12], [q12, q22]])
    mem = lambda P: 0.5 * np.einsum("ij,jk,ik->i", P, Qx, P) + P @ p_e + c_e <= 1
    lam = np.linalg.eigvalsh(Qx)
    frac = convexity_test(mem, n_pairs=800)[2].mean()
    fig, ax = plt.subplots(figsize=(6, 6))
    show_set(ax, mem, f"eig(Q) = {lam.round(2)}  →  {frac:.1%} of segments leave")
    plt.show()

---
### Explain

**1. Which of the sets are convex, and why does the $p = \frac12$ "ball" fail?**

Halfspace (one linear inequality), polyhedron (intersection of halfspaces [CO 3.1, 4.6]), simplex (intersection of the orthant with a hyperplane [CO 4.4]), convex hull (by definition the set of convex combinations), ellipsoid with $Q\succ 0$ (sublevel set of a convex quadratic [CO 3.4, 4.7]) and the norm balls for $p = 1, 2, \infty$ (sublevel sets of norms, which are convex by the triangle inequality [CO 4.8]) are all convex, and they all pass the segment test. The $p = \frac12$ "ball" fails: $\|x\|_{1/2} = (\sqrt{|x_1|} + \sqrt{|x_2|})^2$ is **not a norm** (it violates the triangle inequality: $\|e_1 + e_2\|_{1/2} = 4 > 2 = \|e_1\|_{1/2} + \|e_2\|_{1/2}$), so its unit "ball" is not a sublevel set of a convex function. Concretely, $e_1$ and $e_2$ are on the boundary but their midpoint $(\frac12,\frac12)$ has "norm" 2, so it lies outside: the set is a star with concave edges. Every $p\ge1$ gives a convex ball, every $p<1$ a non-convex one. Affine images of the convex sets stay convex [CO 3.2], as the second test shows.

**2. What happens to the ellipsoid when $Q$ has a negative eigenvalue? (Link to Section 2 of the Linear Algebra portfolio.)**

With an eigen-decomposition $Q = U\Lambda U^\top$ and $z = U^\top x$ (completing the square for $p$), the set is $\sum_i\frac{\lambda_i}{2}z_i^2 \le \text{const}$. With all $\lambda_i > 0$ this is a bounded ellipse (the "bowl" of LA 2.1 cut at a level). If one eigenvalue is negative, the quadratic is a **saddle** (indefinite), and its sublevel set is the region between the two branches of a hyperbola: unbounded along the eigenvector with $\lambda < 0$ and **not convex** (the slider example fails the segment test, e.g. $q_{12} = 2$ gives eigenvalues of opposite sign). With one zero eigenvalue ($Q$ only PSD) the "valley" gives a slab or a parabolic region, which is unbounded but still convex. So the convexity of the sublevel set is decided by the definiteness of $Q$, exactly as the shape of the quadratic form was in LA 2.1.

---
## 3.3 The cone of positive semidefinite matrices

**Task.** Parametrise $2\times2$ symmetric matrices as $X = \begin{bmatrix} x & z\\ z & y\end{bmatrix}$ and plot in 3-D the set $\mathbb S^2_+ = \{x \ge 0,\ y\ge 0,\ xy \ge z^2\}$. Mark a few PSD matrices and draw the segments between them.

### Prediction

- An ice-cream cone with its tip at the origin, opening along the direction $x = y$, $z = 0$ (the identity).
- Every segment between PSD matrices stays inside.

In [ ]:
r = np.linspace(0, 1.2, 40); ph = np.linspace(0, 2 * np.pi, 80)
R, PH = np.meshgrid(r, ph)
# boundary xy = z²: in coordinates t = (x+y)/2, u = (x−y)/2, the condition is u² + z² ≤ t² (a circular cone)
T = R; U = R * np.cos(PH); Z = R * np.sin(PH)
Xb, Yb = T + U, T - U

rng = np.random.default_rng(1)
mats = []
for _ in range(5):
    B = rng.normal(size=(2, 2)) * 0.5
    mats.append(B @ B.T)
pts = np.array([[M[0, 0], M[1, 1], M[0, 1]] for M in mats])

traces = [go.Surface(x=Xb, y=Yb, z=Z, opacity=0.35, colorscale="Blues", showscale=False, name="boundary det X = 0")]
traces.append(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers", marker=dict(size=6, color="red"), name="PSD matrices"))
for i in range(len(pts)):
    for j in range(i + 1, len(pts)):
        traces.append(go.Scatter3d(x=pts[[i, j], 0], y=pts[[i, j], 1], z=pts[[i, j], 2], mode="lines",
                                   line=dict(color="red", width=3), showlegend=False))
fig = go.Figure(traces)
fig.update_layout(scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="cube"), height=600,
                  title="S²₊ = {x ≥ 0, y ≥ 0, xy ≥ z²}")
fig.show()

### Verify numerically

In [ ]:
th = np.linspace(0, 1, 101)
seg_ok = all(np.linalg.eigvalsh(t * mats[i] + (1 - t) * mats[j]).min() >= -1e-12
             for i in range(5) for j in range(5) for t in th)
check("every point on every segment between the PSD matrices is PSD", seg_ok)
check("cone: αX is PSD for α ≥ 0", all(np.linalg.eigvalsh(a * M).min() >= -1e-12 for M in mats for a in [0, 0.5, 3, 100]))
check("boundary surface has det X = xy − z² = 0", np.allclose(Xb * Yb - Z**2, 0))
eig_b = np.array([np.linalg.eigvalsh(np.array([[a, c], [c, b]])) for a, b, c in zip(Xb.ravel(), Yb.ravel(), Z.ravel())])
check("points on the boundary have a zero eigenvalue", np.allclose(eig_b[:, 0], 0, atol=1e-12))

### Explore: the slice $x + y = 1$ ($\mathrm{tr}X = 1$), and the set $\{\det X \ge 1, X\succ0\}$

In [ ]:
u = np.linspace(-1, 1, 401); zz = np.linspace(-0.6, 0.6, 301)
UU, ZZ = np.meshgrid(u, zz)
xs, ys = (1 + UU) / 2, (1 - UU) / 2
psd = (xs >= 0) & (ys >= 0) & (xs * ys >= ZZ**2)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.contourf(UU, ZZ, psd, levels=[0.5, 1.5], colors=["#cfe0f3"]); ax1.contour(UU, ZZ, psd, levels=[0.5], colors="k")
ax1.set_xlabel("u = x − y"); ax1.set_ylabel("z"); ax1.set_aspect("equal")
ax1.set_title("slice tr X = 1:  (x − y)² + 4z² ≤ 1  (a disc in (u, 2z))")

det_set = lambda P: (P[:, 0] > 0) & (P[:, 0] * P[:, 1] - 0.25 > 1)    # slice z = 0.5 of {det X ≥ 1, X ≻ 0}
show_set(ax2, det_set, "slice z = 0.5 of {xy − z² ≥ 1, x > 0}", box=(0, 5))
plt.show()

# the full 3-D set {det X ≥ 1, X ≻ 0}: segment test on 3-D samples
rng = np.random.default_rng(2)
def in_det(P): return (P[:, 0] > 0) & (P[:, 0] * P[:, 1] - P[:, 2] ** 2 >= 1)
S3 = []
while len(S3) < 2000:
    P = rng.uniform([0, 0, -3], [6, 6, 3], size=(20000, 3)); S3.extend(P[in_det(P)])
S3 = np.array(S3[:2000]); S3b = S3[rng.permutation(2000)]
tt = np.linspace(0, 1, 50)[:, None, None]
segs = (tt * S3[None] + (1 - tt) * S3b[None]).reshape(-1, 3)
check("{det X ≥ 1, X ≻ 0} passes the 3-D segment test (2000 pairs)", in_det(segs).all())

---
### Explain

**1. Why is $\mathbb S^n_+$ convex, and why is it a *cone* [CO 4.9]?**

For $X, Y \succeq 0$, $\theta\in[0,1]$ and any vector $v$: $v^\top(\theta X + (1-\theta)Y)v = \theta\,v^\top Xv + (1-\theta)\,v^\top Yv \ge 0$, so the combination is PSD: **convex**. With $\alpha \ge 0$, $v^\top(\alpha X)v = \alpha\,v^\top X v\ge 0$: closed under non-negative scaling, so it is a **cone** (in fact a convex cone, closed under $\alpha X + \beta Y$ with $\alpha,\beta\ge0$). Equivalently, $\mathbb S^n_+ = \bigcap_v\{X : v^\top Xv \ge 0\}$ is an intersection of (infinitely many) halfspaces in the space of symmetric matrices. In the $2\times2$ picture, with $t = \frac{x+y}{2}$, $u = \frac{x-y}{2}$, the conditions become $\sqrt{u^2+z^2}\le t$: a circular "ice-cream" cone around the identity direction.

**2. Which points of your plot have eigenvalue 0?**

The boundary surface $xy = z^2$, i.e. $\det X = 0$ (checked: every boundary point has $\lambda_{\min} = 0$). Interior points have both eigenvalues positive ($X \succ 0$). The tip $X = 0$ has both eigenvalues equal to 0. Each boundary ray is a set of rank-1 matrices $\alpha vv^\top$.

**3. Is the set of matrices with $\det X \ge 1$ (and $X\succ0$) convex?**

**Yes.** On $X\succ0$, $\log\det X$ is concave [CO 8.3] (that is, $-\log\det$ is convex), and $\{\det X \ge 1\} = \{\log\det X \ge 0\}$ is a **superlevel set of a concave function**, which is convex (equivalently, a sublevel set of the convex $-\log\det$). The 3-D segment test agrees, and the $z = 0.5$ slice shows the region above a hyperbola branch $xy \ge 1.25$, which is convex. Without the restriction $X\succ0$ it would not be: $\det X \ge 1$ also contains negative definite matrices ($x, y < 0$), a second disconnected piece.

**4. What shape is the slice $x + y = 1$?**

With $x = \frac{1+u}{2}$, $y = \frac{1-u}{2}$ the condition $xy \ge z^2$ becomes $\frac{1-u^2}{4}\ge z^2$, i.e. $u^2 + (2z)^2 \le 1$: a **disc** (an ellipse in the $(u, z)$ coordinates), centred at $X = \frac12 I$. The set of PSD matrices with trace 1 is the 2×2 "density-matrix" set (the Bloch disc); slicing a circular cone perpendicular to its axis gives a disc, as expected.